#  Run Llama2 in pure Python on your CPU without any third-party packages

### Need three files in the same folder:
1. **Model weights** (stories15M.bin) — a 60 MB Llama 2 architecture checkpoint.
2. **Tokenizer** (tokenizer.bin) — the 32,000-word vocabulary file.
3. **Python script** (run_llama2.py) — the NumPy engine.


### Download Binary files
`curl -L -o stories15M.bin https://huggingface.co`

`stories15M.bin` is a **~60.8 MB** raw binary file that stores the weights of a mini Llama 2 model. 

It starts with a 28-byte header. Right after that, it holds ~15.2 million 32-bit floating-point numbers (`float32`). Unlike `.safetensors` or `.gguf` files, it has no text labels, metadata names, or Python code. It is just a strict list of numbers that a Python script reads in a fixed order.

### Exact Byte-by-Byte Layout of `stories15M.bin`

| Byte Offset | Data Type | Content & Meaning |
| :--- | :--- | :--- |
| **0 – 27** *(28 bytes)* | `7 x int32` | **Model Header (Config):**<br>• `dim = 288` (size of each word vector)<br>• `hidden_dim = 768` (width of the feed-forward layer)<br>• `n_layers = 6` (number of transformer blocks)<br>• `n_heads = 6` (number of attention heads)<br>• `n_kv_heads = 6` (number of key/value heads)<br>• `vocab_size = 32000` (total words/tokens in the vocabulary)<br>• `seq_len = 256` (maximum text length the model can read) |
| **28 – 36,864,027** *(~36.86 MB)* | `float32[32000, 288]` | **`token_embedding` table:**<br>• Turns each of the 32,000 token IDs into a list of 288 numbers.<br>• The model also reuses this table at the end (`wcls`) to pick the next word. |
| **36,864,028 – 60,757,275** *(~23.89 MB)* | `float32[...]` | **6 Transformer Layers** *(repeated 6 times in order)*:<br>• `rms_att` (`6 x 288`) — Weights to normalize attention.<br>• `wq`, `wk`, `wv`, `wo` (`4 x 6 x 288 x 288`) — Matrices for self-attention.<br>• `rms_ffn` (`6 x 288`) — Weights to normalize the feed-forward step.<br>• `w1`, `w2`, `w3` (`3 x 6 x 768 x 288`) — Matrices for the SwiGLU network. |
| **60,757,276 – 60,818,715** *(~61 KB)* | `float32[...]` | **Final Layer & Legacy RoPE Tables:**<br>• `rms_final` (`288`) — Final normalization weights before guessing the next word.<br>• `freq_cis_real` & `freq_cis_imag` (`2 x 256 x 24`) — Precomputed sine and cosine tables for word positions. |

### What Those Numbers Actually Know

* **Trained on TinyStories:** Andrej Karpathy trained this 15M-parameter model on the TinyStories dataset. This dataset contains millions of short children's stories written with simple English grammar.
* **Why it works with Llama 2 code:** This mini model uses 6 layers and a size of 288, while Meta's Llama 2 7B uses 32 layers and a size of 4,096. However, the core math (**RMSNorm**, **RoPE**, **Causal Self-Attention**, and **SwiGLU**) is 100% the same.


In [6]:
!pip install huggingface_hub

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/857.0 kB ? eta -:--:--
   ---------------------------------------- 857.0/857.0 kB 8.7 MB/s  0:00:00
   ---------------------------------------- 0.0/3.9 MB ? eta -:--:--
   ------------------------ --------------- 2.4/3.9 MB 12.3 MB/s eta 0:00:01
   ---------------------------------------- 3.9/3.9 MB 11.3 MB/s  0:00:00

   ---------------------------------------- 0/6 [idna]
   ------ --------------------------------- 1/6 [httpcore2]
   ------ --------------------------------- 1/6 [httpcore2]
   ------ --------------------------------- 1/6 [httpcore2]
   ------ --------------------------------- 1/6 [httpcore2]
   -------------------- ------------------- 3/6 [click]
   -------------------- ------------------- 3/6 [click]
   -------------------- ------------------- 3/6 [click]
   -------------------------- ------------- 4/6 [httpx2]
   -------------------------- ---

In [7]:
from huggingface_hub import hf_hub_download

hf_hub_download(
    repo_id="karpathy/tinyllamas",
    filename="stories15M.bin",
    local_dir="."
)


stories15M.bin: reconstructing file:   0%|          |  0.00B / 60.8MB            

stories15M.bin: downloading bytes:           |  0.00B            

'C:\\Users\\vmanj\\RAG_try\\local_llama\\stories15M.bin'

### Download the binary tokenizer (~430 KB) from GitHub "https://github.com/karpathy/llama2.c/raw/master/tokenizer.bin"


## Inspect the stories15M.bin
You can inspect the exact header integers and floating-point weights inside `stories15M.bin` using a short Python script with struct and `numpy`.

In [14]:
import struct
import numpy as np

filepath = "stories15M.bin"

with open(filepath, "rb") as f:
  # 1. Read the first 28 bytes (7 signed 32-bit integers)
  raw_header = f.read(28)
  header = struct.unpack("7i", raw_header)

  keys = [
      "dim",
      "hidden_dim",
      "n_layers",
      "n_heads",
      "n_kv_heads",
      "vocab_size",
      "seq_len",
  ]
  print("=== 1. RAW HEADER (First 28 Bytes) ===")
  for k, v in zip(keys, header):
    print(f"  {k:<12}: {v}")

  # 2. Read all remaining bytes as 32-bit floats
  weights = np.fromfile(f, dtype=np.float32)

print("\n=== 2. WEIGHTS SUMMARY ===")
print(f"  Total float32 parameters : {weights.size:,}")
print(
    f"  Total file size          : {(28 + weights.size * 4) / (1024**2):.2f} MB"
)
print(
    f"  Min / Max weight value   : {weights.min():.6f} / {weights.max():.6f}"
)
print(
    f"  Mean / Std deviation     : {weights.mean():.6f} / {weights.std():.6f}"
)

# 3. Slice out specific tensors to inspect actual numbers
dim = header[0]
vocab_size = abs(header[5])
token_embedding = weights[: vocab_size * dim].reshape(vocab_size, dim)

print("\n=== 3. SAMPLE RAW WEIGHTS ===")
print("  First 10 numbers of Token ID 1 (BOS) embedding vector:")
print(" ", np.round(token_embedding[1, :10], 5))

print("\n  First 10 numbers of Token ID 2 (EOS) embedding vector:")
print(" ", np.round(token_embedding[2, :10], 5))



=== 1. RAW HEADER (First 28 Bytes) ===
  dim         : 288
  hidden_dim  : 768
  n_layers    : 6
  n_heads     : 6
  n_kv_heads  : 6
  vocab_size  : 32000
  seq_len     : 256

=== 2. WEIGHTS SUMMARY ===
  Total float32 parameters : 15,204,000
  Total file size          : 58.00 MB
  Min / Max weight value   : -1.000000 / 9.860225
  Mean / Std deviation     : 0.002921 / 0.055608

=== 3. SAMPLE RAW WEIGHTS ===
  First 10 numbers of Token ID 1 (BOS) embedding vector:
  [-0.0171  -0.01353  0.00674 -0.00455  0.03713  0.01478 -0.0373   0.01211
  0.01862  0.00697]

  First 10 numbers of Token ID 2 (EOS) embedding vector:
  [-0.05983 -0.01456  0.10616  0.02165 -0.00434 -0.00697  0.00942  0.0291
  0.02411  0.01985]


# 🔍 Inspection Breakdown of `stories15M.bin`

This output reveals the exact architecture blueprint and raw learned parameters stored inside the `stories15M.bin` Llama 2 model checkpoint.

---

## 1. Raw Header (First 28 Bytes)
The first 28 bytes of the file store 7 integers (`int32`) that define the model's shape and capacity:

* **`dim: 288` (Embedding Dimension):** Every word/token is represented internally as a list (vector) of **288 numbers**. *(For comparison, Llama 2 7B uses `4096`.)*
* **`hidden_dim: 768` (Feed-Forward Width):** Inside each layer's SwiGLU neural network block, the 288-number vector expands out to **768 numbers** to process complex patterns before shrinking back to 288.
* **`n_layers: 6` (Transformer Blocks):** The input passes sequentially through **6 stacked transformer layers** from bottom to top. *(Llama 2 7B uses `32` layers.)*
* **`n_heads: 6` (Attention Heads):** Each layer splits the 288 dimensions into **6 parallel attention heads** (`288 ÷ 6 = 48` dimensions per head), allowing the model to focus on 6 different grammatical or contextual relationships at once.
* **`n_kv_heads: 6` (Key/Value Heads):** Matches `n_heads` (`6`), meaning this model uses standard **Multi-Head Attention (MHA)** rather than Grouped-Query Attention (GQA).
* **`vocab_size: 32000` (Vocabulary Size):** The model knows **32,000 unique tokens** (words, sub-words, punctuation, and special control tokens).
* **`seq_len: 256` (Context Window):** The maximum number of tokens the model can read and remember at one time is **256 tokens** (~180–200 English words).

---

## 2. Weights Summary
After the 28-byte header, the rest of the file is one continuous array of 32-bit floating-point numbers (`float32`):

* **`Total float32 parameters: 15,204,000`:** The model has **~15.2 million trainable parameters** (the "15M" in `stories15M.bin`).
* **`Total file size: 58.00 MB`:** Because each `float32` number takes **4 bytes** of storage:
  $$\text{15,204,000 parameters} \times \text{4 bytes} + \text{28 header bytes} = \text{60,816,028 bytes} \approx \text{58.00 MiB}$$
* **`Min / Max weight value: -1.000000 / 9.860225`:** Most matrix weights stay small (around `-1.0` to `+1.0`), while a few scaling parameters in `RMSNorm` layers reach up to `9.86` to amplify important signals.
* **`Mean / Std deviation: 0.002921 / 0.055608`:** The weights are tightly clustered around **zero** (`mean ≈ 0.003`, `std ≈ 0.056`), which keeps math stable across all 6 layers without numbers exploding or vanishing.

---

## 3. Sample Raw Weights
These arrays show the first 10 dimensions (out of 288) for two special control tokens in the `token_embedding` table:

* **Token ID 1 (`BOS` — Beginning of Sequence):**
  `[-0.0171, -0.01353, 0.00674, -0.00455, 0.03713, ...]`
  This 288-number coordinate vector signals to the transformer that a **new story is starting**.
* **Token ID 2 (`EOS` — End of Sequence):**
  `[-0.05983, -0.01456, 0.10616, 0.02165, -0.00434, ...]`
  When the model predicts this token at the output, the generation loop knows the **story is finished** and stops generating.


In [15]:
import struct

filepath = "tokenizer.bin"
vocab_size = 32000  # From stories15M.bin header

vocab = []
scores = []

with open(filepath, "rb") as f:
  # 1. First 4 bytes: maximum token character length
  max_token_length = struct.unpack("i", f.read(4))[0]

  # 2. Loop 32,000 times: [float32 score][int32 byte_length][UTF-8 bytes]
  for _ in range(vocab_size):
    score, length = struct.unpack("fi", f.read(8))
    token_bytes = f.read(length)
    vocab.append(token_bytes.decode("utf-8", errors="replace"))
    scores.append(score)

print("=== 1. TOKENIZER SUMMARY ===")
print(f"  Total tokens     : {len(vocab):,}")
print(f"  Max token length : {max_token_length} bytes")

print("\n=== 2. SPECIAL CONTROL TOKENS (IDs 0–2) ===")
for tid in range(3):
  print(f"  ID {tid:<2} | Score: {scores[tid]:<8.2f} | Token: {repr(vocab[tid])}")

print("\n=== 3. SAMPLE FULL WORDS (IDs 500–509) ===")
for tid in range(500, 510):
  # '▁' (\u2581) marks a leading space before a word
  clean = vocab[tid].replace("\u2581", "␣")
  print(f"  ID {tid:<5} | Score: {scores[tid]:<8.2f} | Token: {clean}")

print("\n=== 4. LOOK UP SPECIFIC WORDS ===")
for word in ["\u2581Once", "\u2581cat", "\u2581robot", "ing"]:
  if word in vocab:
    tid = vocab.index(word)
    print(f"  {word.replace(chr(0x2581), '␣'):<10} -> ID {tid} (score: {scores[tid]:.2f})")
  else:
    print(f"  {word.replace(chr(0x2581), '␣'):<10} -> Not a single token (split into subwords)")


=== 1. TOKENIZER SUMMARY ===
  Total tokens     : 32,000
  Max token length : 27 bytes

=== 2. SPECIAL CONTROL TOKENS (IDs 0–2) ===
  ID 0  | Score: 0.00     | Token: '<unk>'
  ID 1  | Score: 0.00     | Token: '\n<s>\n'
  ID 2  | Score: 0.00     | Token: '\n</s>\n'

=== 3. SAMPLE FULL WORDS (IDs 500–509) ===
  ID 500   | Score: -241.00  | Token:  }
  ID 501   | Score: -242.00  | Token:  U
  ID 502   | Score: -243.00  | Token:  us
  ID 503   | Score: -244.00  | Token:  z
  ID 504   | Score: -245.00  | Token: ust
  ID 505   | Score: -246.00  | Token:  have
  ID 506   | Score: -247.00  | Token: lic
  ID 507   | Score: -248.00  | Token: ни
  ID 508   | Score: -249.00  | Token:  can
  ID 509   | Score: -250.00  | Token: tr

=== 4. LOOK UP SPECIFIC WORDS ===
  ␣Once      -> Not a single token (split into subwords)
  ␣cat       -> Not a single token (split into subwords)
  ␣robot     -> Not a single token (split into subwords)
  ing        -> ID 292 (score: -33.00)


In [16]:
import os
import struct
import time
import numpy as np


# --- 1. VECTORIZED MATH HELPERS ---
def rmsnorm(x, weight):
  ss = np.mean(x**2) + 1e-5
  return (x / np.sqrt(ss)) * weight


def softmax(x):
  e_x = np.exp(x - np.max(x))
  return e_x / np.sum(e_x)

In [17]:

# --- 2. MODEL & TOKENIZER LOADERS ---
def load_checkpoint(filepath):
  if not os.path.exists(filepath):
    raise FileNotFoundError(f'Model file not found: {filepath}')

  with open(filepath, 'rb') as f:
    # Read 7 integers (28 bytes) header
    header = struct.unpack('7i', f.read(28))
    dim, hidden_dim, n_layers, n_heads, n_kv_heads, vocab_size, seq_len = header
    shared_weights = vocab_size > 0
    vocab_size = abs(vocab_size)

    config = {
        'dim': dim,
        'hidden_dim': hidden_dim,
        'n_layers': n_layers,
        'n_heads': n_heads,
        'n_kv_heads': n_kv_heads,
        'vocab_size': vocab_size,
        'seq_len': seq_len,
    }

    # Load all float32 weights into RAM in one fast operation
    data = np.fromfile(f, dtype=np.float32)
    offset = 0

    def take(shape):
      nonlocal offset
      size = int(np.prod(shape))
      arr = data[offset : offset + size].reshape(shape)
      offset += size
      return arr

    kv_dim = (dim * n_kv_heads) // n_heads
    weights = {
        'token_embedding': take((vocab_size, dim)),
        'rms_att': take((n_layers, dim)),
        'wq': take((n_layers, dim, dim)),
        'wk': take((n_layers, kv_dim, dim)),
        'wv': take((n_layers, kv_dim, dim)),
        'wo': take((n_layers, dim, dim)),
        'rms_ffn': take((n_layers, dim)),
        'w1': take((n_layers, hidden_dim, dim)),
        'w2': take((n_layers, dim, hidden_dim)),
        'w3': take((n_layers, hidden_dim, dim)),
        'rms_final': take((dim,)),
    }
    # Skip legacy RoPE frequency tables if present
    offset += seq_len * (dim // n_heads)
    weights['wcls'] = (
        weights['token_embedding']
        if shared_weights
        else take((vocab_size, dim))
    )

  return config, weights


def load_tokenizer(filepath, vocab_size):
  if not os.path.exists(filepath):
    raise FileNotFoundError(f'Tokenizer file not found: {filepath}')

  vocab, scores = [], []
  with open(filepath, 'rb') as f:
    _ = struct.unpack('i', f.read(4))  # max_token_length
    for _ in range(vocab_size):
      score, length = struct.unpack('fi', f.read(8))
      token_bytes = f.read(length)
      vocab.append(token_bytes.decode('utf-8', errors='replace'))
      scores.append(score)
  return vocab, scores


def bpe_encode(text, vocab, scores):
  # SentencePiece represents spaces with '▁' (\u2581)
  text = ' ' + text.replace(' ', '\u2581')
  vocab_map = {token: i for i, token in enumerate(vocab)}

  tokens = [1]  # 1 = BOS (Beginning of Sequence)
  for ch in text:
    if ch in vocab_map:
      tokens.append(vocab_map[ch])

  while True:
    best_score = -1e10
    best_id = -1
    best_idx = -1
    for i in range(len(tokens) - 1):
      merged = vocab[tokens[i]] + vocab[tokens[i + 1]]
      tid = vocab_map.get(merged, -1)
      if tid != -1 and scores[tid] > best_score:
        best_score = scores[tid]
        best_id = tid
        best_idx = i
    if best_idx == -1:
      break
    tokens[best_idx] = best_id
    tokens.pop(best_idx + 1)

  return tokens


In [18]:


# --- 3. LLAMA 2 TRANSFORMER FORWARD PASS ---
def llama2_forward(token, pos, config, weights, k_cache, v_cache):
  dim = config['dim']
  n_heads = config['n_heads']
  n_kv_heads = config['n_kv_heads']
  head_dim = dim // n_heads
  kv_mul = n_heads // n_kv_heads

  x = weights['token_embedding'][token].copy()

  freqs = 1.0 / (
      10000.0 ** (np.arange(0, head_dim, 2, dtype=np.float32) / head_dim)
  )
  angles = pos * freqs
  cos_a, sin_a = np.cos(angles), np.sin(angles)

  for l in range(config['n_layers']):
    xb = rmsnorm(x, weights['rms_att'][l])

    q = (weights['wq'][l] @ xb).reshape(n_heads, head_dim)
    k = (weights['wk'][l] @ xb).reshape(n_kv_heads, head_dim)
    v = (weights['wv'][l] @ xb).reshape(n_kv_heads, head_dim)

    # Apply RoPE positional rotation
    q0, q1 = q[:, 0::2], q[:, 1::2]
    q[:, 0::2], q[:, 1::2] = q0 * cos_a - q1 * sin_a, q0 * sin_a + q1 * cos_a

    k0, k1 = k[:, 0::2], k[:, 1::2]
    k[:, 0::2], k[:, 1::2] = k0 * cos_a - k1 * sin_a, k0 * sin_a + k1 * cos_a

    k_cache[l, pos] = k
    v_cache[l, pos] = v

    keys = np.repeat(k_cache[l, : pos + 1], kv_mul, axis=1)
    vals = np.repeat(v_cache[l, : pos + 1], kv_mul, axis=1)

    scores = np.einsum('hd,thd->ht', q, keys) / np.sqrt(head_dim)
    probs = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
    probs /= np.sum(probs, axis=-1, keepdims=True)

    xb_att = np.einsum('ht,thd->hd', probs, vals).reshape(dim)
    x += weights['wo'][l] @ xb_att

    # SwiGLU Feed-Forward Network
    xb = rmsnorm(x, weights['rms_ffn'][l])
    w1 = weights['w1'][l] @ xb
    w3 = weights['w3'][l] @ xb
    silu = w1 * (1.0 / (1.0 + np.exp(-w1)))
    x += weights['w2'][l] @ (silu * w3)

  x = rmsnorm(x, weights['rms_final'])
  return weights['wcls'] @ x



In [19]:

# --- 4. RUNNER ---
def generate(
    model_path, tokenizer_path, prompt, max_tokens=150, temperature=0.8
):
  print(f'Loading {model_path}...')
  config, weights = load_checkpoint(model_path)
  vocab, scores = load_tokenizer(tokenizer_path, config['vocab_size'])
  print(
      f"Loaded! Layers: {config['n_layers']}, Dim: {config['dim']}, Context:"
      f" {config['seq_len']}\n"
  )

  prompt_tokens = bpe_encode(prompt, vocab, scores)
  head_dim = config['dim'] // config['n_heads']

  k_cache = np.zeros(
      (config['n_layers'], config['seq_len'], config['n_kv_heads'], head_dim),
      dtype=np.float32,
  )
  v_cache = np.zeros_like(k_cache)

  start_time = time.time()
  tokens_generated = 0
  token = prompt_tokens[0]

  for pos in range(min(config['seq_len'], len(prompt_tokens) + max_tokens)):
    logits = llama2_forward(token, pos, config, weights, k_cache, v_cache)

    if pos + 1 < len(prompt_tokens):
      next_token = prompt_tokens[pos + 1]
    else:
      if temperature == 0.0:
        next_token = int(np.argmax(logits))
      else:
        probs = softmax(logits / temperature)
        next_token = int(np.random.choice(len(probs), p=probs))
      tokens_generated += 1

    if next_token == 2:  # EOS token
      break

    piece = vocab[next_token].replace('\u2581', ' ')
    print(piece, end='', flush=True)
    token = next_token

  elapsed = time.time() - start_time
  print(
      f'\n\n--- Generated {tokens_generated} tokens in {elapsed:.2f}s'
      f' ({tokens_generated / max(elapsed, 1e-5):.1f} tok/s) ---'
  )




In [20]:
if __name__ == '__main__':
  generate(
      model_path='stories15M.bin',
      tokenizer_path='tokenizer.bin',
      prompt='Once upon a time, a curious cat found a glowing key',
      max_tokens=150,
      temperature=0.8,
  )


Loading stories15M.bin...
Loaded! Layers: 6, Dim: 288, Context: 256

 Onceuponatime,acuriouscatfoundaglowingkey, a small girl. She was so excited to play with her new toy. She grabbed it and shouted, "Look, I'm gonna show you my new toy!"<0x0A>The older girl said, "No way, it's not safe for you to play with it, so it's best to leave it alone." But contest was too tempting, so the little girl decided to make a game out of it. She said that the other kids would agree.<0x0A>The first kid said, "Let's all play together." The older kid said, "OK, then you can take it inside and play with it, but please don't do anything dangerous!" The younger kid said, "

--- Generated 151 tokens in 2.56s (59.0 tok/s) ---
